In [12]:

import os
import re
from pathlib import Path
from fnmatch import fnmatch
from subprocess import run
from typing import Literal, TypedDict, Annotated

import requests
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

load_dotenv(override=True)


@tool(parse_docstring=True)
def read(path: str, offset: int = 1, limit: int = 200) -> str:
    """读取文件内容，大文件用 offset/limit 翻页。

    Args:
        path: 文件路径。
        offset: 起始行号（从 1 开始）。
        limit: 最多读取的行数。

    Returns:
        文件内容；未读完时末尾附续读提示。
    """
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    if offset > len(lines):
        return f"offset 超出文件末尾（共 {len(lines)} 行）"
    start = max(offset - 1, 0)
    end = min(start + limit, len(lines))
    text = "\n".join(lines[start:end])
    if len(text) > 20_000:
        text = text[:20_000] + "\n...[单次输出过长已截断，请减小 limit]"
    if end < len(lines):
        text += f"\n\n[共 {len(lines)} 行，已读到第 {end} 行，用 offset={end + 1} 续读]"
    return text


@tool(parse_docstring=True)
def ls(path: str = ".") -> str:
    """列出目录内容。

    Args:
        path: 目录路径，默认当前目录。

    Returns:
        每行一项，目录以 / 结尾；超长截断。
    """
    p = Path(path)
    if not p.is_dir():
        return f"不是目录：{path}"
    items = sorted(p.iterdir(), key=lambda x: (not x.is_dir(), x.name))
    out = "\n".join(f"{i.name}/" if i.is_dir() else i.name for i in items)
    return out[:10_000] + ("\n...[已截断]" if len(out) > 10_000 else "")


@tool(parse_docstring=True)
def grep(pattern: str, path: str = ".", glob: str = "*") -> str:
    """按正则搜索文件内容。

    Args:
        pattern: 正则表达式。
        path: 要搜索的文件或目录，默认当前目录。
        glob: 目录模式下只搜匹配该文件名的文件，如 "*.py"。

    Returns:
        匹配行，格式 `文件:行号:内容`，最多 200 条；无匹配返回提示。
    """
    base = Path(path)
    if base.is_file():
        files = [base]
    else:
        skip = {".git", "__pycache__", "node_modules", ".venv"}
        files = []
        for root, dirs, names in os.walk(base):
            dirs[:] = [d for d in dirs if d not in skip]
            files += [Path(root) / n for n in names if fnmatch(n, glob)]
    hits = []
    for f in files:
        try:
            text = f.read_text(encoding="utf-8")
        except (UnicodeDecodeError, PermissionError, OSError):
            continue
        for n, line in enumerate(text.splitlines(), 1):
            if re.search(pattern, line):
                hits.append(f"{f}:{n}:{line.strip()}")
        if len(hits) >= 200:
            hits = hits[:200] + ["...[已达上限，可缩小 path/glob]"]
            break
    return "\n".join(hits) or "(无匹配)"


@tool(parse_docstring=True)
def write(path: str, content: str) -> str:
    """写入/覆盖文件，自动创建父目录。

    Args:
        path: 文件路径。
        content: 要写入的文本内容。

    Returns:
        写入结果提示信息。
    """
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return f"已写入 {len(content)} 字符到 {path}"


@tool(parse_docstring=True)
def shell(cmd: str) -> str:
    """执行 shell 命令。

    Args:
        cmd: 要执行的命令。

    Returns:
        stdout 与 stderr 合并后的输出，超长截断；非零退出码附在末尾。
    """
    r = run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    out = (r.stdout + r.stderr).strip() or "(无输出)"
    if len(out) > 10_000:
        out = out[:10_000] + "\n...[输出过长已截断]"
    if r.returncode != 0:
        out += f"\n[退出码: {r.returncode}]"
    return out


@tool(parse_docstring=True)
def web_search(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


@tool(parse_docstring=True)
def tavily_search(
        query: str,
        max_results: int = 2,
        search_depth: Literal["basic", "advanced"] = "basic",
        topic: Literal["general", "news", "finance"] = "general",
        time_range: Literal["day", "week", "month", "year"] | None = None,
        timeout: int = 30,
) -> str:
    """联网搜索。

    Args:
        query: 搜索关键词。
        max_results: 返回结果条数上限。
        search_depth: 搜索深度，"advanced" 更精准但更慢。
        topic: 搜索类别，"news" 用于新闻时事。
        time_range: 时效过滤，只保留近期结果；None 表示不过滤。
        timeout: 请求超时秒数。

    Returns:
        Tavily 原始响应。
    """
    r = requests.post(
        "https://api.tavily.com/search",
        headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
        json={
            "query": query,
            "max_results": max_results,
            "search_depth": search_depth,
            "topic": topic,
            "time_range": time_range,
        },
        timeout=timeout,
    )
    return str(r.json())

In [13]:

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode
from langgraph.constants import END, START
from langchain.chat_models import init_chat_model
from langgraph.graph import add_messages, StateGraph

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
)

explore_tools = [read, ls, grep, tavily_search]
model_explore = model.bind_tools(explore_tools)


# 子图状态
class ExploreState(TypedDict):
    messages: Annotated[list, add_messages]


# 子图大模型节点
def llm_explore(state: ExploreState) -> ExploreState:
    ai_msg = model_explore.invoke(state["messages"])
    return {"messages": [ai_msg]}


# 子图工具节点
explore_tool_node = ToolNode(explore_tools, handle_tool_errors=True)


# 子图路由函数
def sub_router(state: ExploreState) -> Literal["explore_tool_node", END]:
    return "explore_tool_node" if state["messages"][-1].tool_calls else END


sub_builder = StateGraph(ExploreState)
sub_builder.add_node("llm_explore", llm_explore)
sub_builder.add_node("explore_tool_node", explore_tool_node)
sub_builder.add_edge(START, "llm_explore")
sub_builder.add_conditional_edges("llm_explore", sub_router)
sub_builder.add_edge("explore_tool_node", "llm_explore")
explore_graph = sub_builder.compile()

In [15]:
# ---- 父图：持有全量工具，把 explore 子图包装成 research 工具（sub-agent as tool）----
EXPLORE_PROMPT = (
    "你是代码探索员，只做只读调查，禁止修改/创建/删除文件。"
    "用工具收集信息，完成后输出结构化摘要，不再调用工具。"
)


@tool(parse_docstring=True)
def research(task: str) -> str:
    """只读调研，返回结构化摘要。适合多步骤的深度调查；零散小问题自己查即可。

    Args:
        task: 要调查的问题，需自包含（文件路径、目标等）。

    Returns:
        调研结论的结构化摘要。
    """
    result = explore_graph.invoke(
        {"messages": [SystemMessage(EXPLORE_PROMPT), HumanMessage(task)]}
    )
    return result["messages"][-1].content


agent_tools = [research, read, ls, grep, write, shell, web_search, tavily_search]
model_agent = model.bind_tools(agent_tools)


# 父图状态
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 父图大模型节点
def llm_agent(state: AgentState) -> AgentState:
    ai_msg = model_agent.invoke(state["messages"])
    return {"messages": [ai_msg]}


# 父图工具节点
agent_tool_node = ToolNode(agent_tools, handle_tool_errors=True)


# 父图路由函数
def agent_router(state: AgentState) -> Literal["agent_tool_node", END]:
    return "agent_tool_node" if state["messages"][-1].tool_calls else END


parent_builder = StateGraph(AgentState)
parent_builder.add_node("llm_agent", llm_agent)
parent_builder.add_node("agent_tool_node", agent_tool_node)
parent_builder.add_edge(START, "llm_agent")
parent_builder.add_conditional_edges("llm_agent", agent_router)
parent_builder.add_edge("agent_tool_node", "llm_agent")
parent_graph = parent_builder.compile()

res = parent_graph.invoke(
    {
        "messages": [
            HumanMessage(
                "先调研当前目录有哪些 ipynb 文件、00_工具列表.ipynb 定义了哪几个工具，"
                "然后把调研结论写入当前目录下的 research_report.md"
            )
        ]
    }
)
print(res)

{
    'messages': [
        HumanMessage(
            content='先调研当前目录有哪些 ipynb 文件、00_工具列表.ipynb 
定义了哪几个工具，然后把调研结论写入当前目录下的 research_report.md',
            additional_kwargs={},
            response_metadata={},
            id='060e503c-af87-41ba-916a-2d691e87c504'
        ),
        AIMessage(
            content="I'll start by exploring the current directory structure.",
            additional_kwargs={'refusal': None, 'reasoning_content': ''},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 1102,
                    'total_tokens': 1192,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 896,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 896,
                    'prompt_cache_miss_tokens': 206
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '6967dac0-3eb1-4b10-872d-d8bb371ce87d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0c006-3925-7092-afac-e18743c5db75-0',
            tool_calls=[
                {
                    'name': 'ls',
                    'args': {'path': '.'},
                    'id': 'call_00_ET_PohUgrzQdBeAhaRD2phi0657',
                    'type': 'tool_call'
                },
                {
                    'name': 'grep',
                    'args': {'pattern': 'ipynb', 'path': '.'},
                    'id': 'call_01_ET_Im5mq44n0l9mzUuQ89Ky1586',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1102,
                'output_tokens': 90,
                'total_tokens': 1192,
                'input_token_details': {'cache_read': 896},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='00_工具列表.ipynb\n01_explore子图.ipynb\nCheatSheet.md\nresearch_report.md',
            name='ls',
            id='bc152233-d996-4353-9cb6-51aedb30eb5b',
            tool_call_id='call_00_ET_PohUgrzQdBeAhaRD2phi0657'
        ),
        ToolMessage(
            content='01_explore子图.ipynb:282:"                \\"先调研当前目录有哪些 ipynb 
文件、00_工具列表.ipynb 定义了哪几个工具，\\"\\n",\n01_explore子图.ipynb:297:"## 一、当前目录的 ipynb 
文件\\n",\n01_explore子图.ipynb:299:"实测 `dir \\u001B[35m/\\u001B[0m\\u001B[95ma\\u001B[0m 
\\u001B[35m/\\u001B[0m\\u001B[95mb\\u001B[0m`：目录共 **\\u001B[1;36m4\\u001B[0m 
个文件、无子目录、无隐藏文件**。ipynb 共 **\\u001B[1;36m2\\u001B[0m 个**：\\n",\n01_explore子图.ipynb:303:"| 
`00_工具列表.ipynb` | \\u001B[1;36m8\\u001B[0m,\\u001B[1;36m474\\u001B[0m B | 工具定义（`@tool` 集中定义） 
|\\n",\n01_explore子图.ipynb:304:"| `01_explore子图.ipynb` | \\u001B[1;36m12\\u001B[0m,\\u001B[1;36m943\\u001B[0m B
| 把 explore 子图包装成 `research` 工具挂到父图 |\\n",\n01_explore子图.ipynb:306:"另两个非 
ipynb：`CheatSheet.md`、`research_report.md`（本报告）。\\n",\n01_explore子图.ipynb:310:"## 二、`00_工具列表.ipynb`
定义了哪几个工具\\n",\n01_explore子图.ipynb:332:"（第 \\u001B[1;36m126\\u001B[0m 行 \\u001B[1;36m3\\u001B[0m 个 
ipynb 缺失的结论我已在报告中保留并核实。）\\n"\n01_explore子图.ipynb:337:"## 一、当前目录的 ipynb 
文件\\n",\n01_explore子图.ipynb:339:"实测 `dir <span style=\\"color: #800080; text-decoration-color: 
#800080\\">/</span><span style=\\"color: 